# Challenge 3 · Climate

[Start Here](../../Start_Here.ipynb) · Previous: [Flow](../02_fluid/Challenge_2_Fluid_Flow.ipynb) · Next: [Neural Operators](../04_neural_operators/Challenge_4_Neural_Operators.ipynb)

Learn how transport, diffusion, and heat exchange affect temperature. Level 1 predicts one temperature field from $(x,y,t)$. Level 2 implements two fields and their heat-exchange terms; its original baseline has zero coupling, followed by an optional coupled experiment. Both simplified models have analytic reference solutions under the default settings.

Write residuals, coefficients, initial/boundary targets, and baseline analytic solutions. Check the heat-exchange signs and compare predictions with an independently supplied reference.

### Run this Challenge

1. Run setup in student mode (`USE_REFERENCE = False`). An unfinished exercise should stop with a message, not show a completed answer. For a demonstration only, explicitly set `True`; this bypasses your edits.
2. Complete `student_equations`, `student_parameters`, `student_conditions`, and `student_solution` in the linked `.py` file. Save with Ctrl+S / Command+S; editing an example in this notebook does not change the program.
3. Run the level's training cell, then its result cell. Each attempt gets a fresh directory. After changing code, settings or mode, rerun both cells.
4. Set `STEPS = 2` and `DEVICE = "cpu"` for a quick execution check. An execution check does not establish convergence. Choose longer runs using the held-out errors and predictions below.

The code uses PhysicsNeMo 2.2.2: a `FullyConnected` network, a SymPy `PDE`, `PhysicsInformer`, and a PyTorch optimizer.

### Practice, demonstration, submission

Local plots are practice feedback, not submitted scores. Register your nickname in **Submit your code** below, then use that panel to send your saved implementation and view the judge's results in this notebook. Instructor demonstrations cannot be submitted.


In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "02_challenges" / "03_climate" / "climate_l1.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings

LAB_DIR = ROOT / "02_challenges" / "03_climate"
USE_REFERENCE = False  # Student mode. Set True only for an instructor demonstration.
# Server AI4SCI_REFERENCE settings do not change this student default.
JUDGE_URL = os.environ.get("AI4SCI_JUDGE_URL", "")  # Optional API URL; private launch configuration is also supported.
DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")
STEPS = int(os.environ.get("AI4SCI_STEPS", "10000"))  # Class budget; compare each field against its analytic solution.
SEED = 42
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB_DIR / "outputs"))).expanduser().resolve()
RUN_DIRS = {}  # Latest attempt per level; results are never shared between levels.
RUN_COMPLETED = {}

def show_mode():
    validate_settings(DEVICE, STEPS, USE_REFERENCE)
    print("Mode: INSTRUCTOR REFERENCE; student_* edits are bypassed." if USE_REFERENCE
          else "Mode: STUDENT; saved student_* functions will run.")

show_mode()
print({"device": DEVICE, "steps": STEPS, "output_base": str(OUTPUT_BASE)})

from ETC.runtime.submission import show_submission_panel, show_submission_controls
show_submission_panel("3", reference=USE_REFERENCE, judge_url=JUDGE_URL)


## Level 1 · Temperature Transport and Diffusion

This temperature advection–diffusion–reaction problem uses the spatial domain $[0,\pi]^2$ and time interval $[0,2\pi]$.
$$T_t+u_0T_x+v_0T_y-\kappa\Delta T-Q_0+\lambda(T-T_{eq})=0.$$
The initial value is $\sin x\sin y$, and the temperature is zero on all four edges. The defaults $u_0=v_0=Q_0=\lambda=0,\kappa=1$ reduce the equation to diffusion.
For zero advection, source, and relaxation, the analytic solution is $T=\sin x\sin y\,e^{-2\kappa t}$. The code checks those assumptions before reporting reference errors. For a separate local experiment, change `kappa` in both `student_parameters` and [config_atmos.yaml](conf/config_atmos.yaml), so training and the comparison describe the same problem. This changes the decay rate; introducing the other terms invalidates this reference.

### Code and exercise

Complete `student_equations` in [climate_l1.py](climate_l1.py), returning the residual under the key `adr`. Include the terms whose default coefficients are zero: the default diffusion run cannot reveal a missing advection, source, or relaxation term. Also complete `student_parameters`, `student_conditions`, and `student_solution`. Return `initial_T` and `boundary_T`; derive the baseline analytic expression under key `T`. Your expression is checked, not used as the ground truth for scoring. Follow your conditions through `loss_terms`. `PhysicsInformer` computes spatial derivatives; PyTorch autograd supplies the time derivative.

Before running, predict what doubling `kappa` does to the decay. After the baseline, change only that coefficient and compare the predictions with the corresponding analytic solution. Nonzero advection, sources, or relaxation are extensions; they need another independent reference.


In [ ]:
RUN_COMPLETED[1] = False
result_dir = OUTPUT_BASE / f"climate_l1-{uuid4().hex}"
RUN_DIRS[1] = result_dir
command = [sys.executable, str(LAB_DIR / "climate_l1.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[1] = True


### Read the temperature result

Compare the prediction with the sine-decay solution, then inspect the PDE and initial/boundary errors separately. The preview is one time slice, not a check of the whole trajectory. `reference_over_time` compares the solution at five fixed times, including the initial and final times. Look at both RMSE and relative error: late in diffusion, a small absolute error can hide a poor prediction because the true temperature is already close to zero.

The first and last held-out rows in `loss.csv` use identical points and the provided PDE. Intermediate rows use freshly sampled minibatches and your training equation, so compare the held-out rows with each other. `metrics.json` reports the errors, `model.pt` stores the model and configuration, and `predictions.npz` contains the temperature field.


In [ ]:
if not RUN_COMPLETED.get(1, False):
    raise RuntimeError("The current Level 1 training run has not completed.")
result_dir = RUN_DIRS[1]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Level 2 · Coupled Atmosphere–Ocean System

Predict the atmospheric temperature $T_a$ and ocean temperature $T_o$ together on the same space-time domain.
$$T_{a,t}+u_0T_{a,x}+v_0T_{a,y}-\kappa_a\Delta T_a-Q_a+\lambda_a(T_a-T_{eq,a})+\gamma(T_a-T_o)=0,$$
$$T_{o,t}-\kappa_o\Delta T_o-Q_o-\gamma(T_a-T_o)=0.$$
The exchange terms have opposite signs, so internal heat exchange cancels when the two equations are added. Both initial values are $\sin x\sin y$, and both boundary values are zero.
The original baseline in [config_coupled.yaml](conf/config_coupled.yaml) sets `gamma0: 0.0`. Implement the exchange terms even though this baseline cannot activate them; the symbolic submission check uses a nonzero symbolic coefficient. The diffusivities are $\kappa_a=1$ and $\kappa_o=0.5$; advection, sources, and relaxation are zero. The fields start at the same temperature but cool at different rates. In the optional `gamma0=0.5` experiment, exchange transfers heat from the warmer field to the cooler one.

For an independent reference, write $T_a=a(t)\sin x\sin y$ and $T_o=o(t)\sin x\sin y$. Their amplitudes satisfy
$$a'=-(2\kappa_a+\gamma)a+\gamma o,\qquad o'=\gamma a-(2\kappa_o+\gamma)o,\qquad a(0)=o(0)=1.$$
`exact_reference` solves the coupled linear system by matrix eigendecomposition. It also handles `gamma0 = 0`. The reference is disabled when advection, sources, or relaxation are enabled.

### Code and exercise

Complete both residuals in `student_equations` in [climate_l2.py](climate_l2.py), using keys `atmosphere` and `ocean`. Add the equations on paper to check that the exchange terms cancel. This cancellation concerns internal exchange only; boundary losses and external sources can still change the total heat. Complete `student_parameters` with the stated baseline, `student_conditions` with `initial_Ta`, `initial_To`, `boundary_Ta`, `boundary_To`, and `student_solution` with the two uncoupled sine-decay solutions. Each field needs its own contributions; all these functions are submitted.

After completing the baseline, keep all other settings fixed and compare `gamma0: 0.0` with `gamma0: 0.5`. In student mode update both `student_parameters` and the matching YAML coefficient for that separate local experiment; the independent coupled reference follows the YAML. Predict which field warms relative to the uncoupled solution and which cools. Save both configurations with their results, then restore the default for like-for-like comparisons.


In [ ]:
RUN_COMPLETED[2] = False
result_dir = OUTPUT_BASE / f"climate_l2-{uuid4().hex}"
RUN_DIRS[2] = result_dir
command = [sys.executable, str(LAB_DIR / "climate_l2.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[2] = True


### Compare the two temperatures

Inspect both fields, not just the total loss. With these diffusivities, the atmosphere cools faster in the uncoupled case. Exchange therefore raises its temperature relative to that baseline and lowers the ocean temperature. Do the two predictions and their analytic references show this change? `reference_over_time.per_field` reports separate errors for `Ta` and `To` across the five evaluation times; `per_time` reports their combined error at each time. Also inspect all six held-out PDE/initial/boundary contributions. A decreasing training loss alone does not show that the exchange signs are correct.

### Practice evaluation

Compare reference and condition errors with the problem and evaluation settings held fixed. Evaluation uses the provided equations even when you edit the training residual. These local results are practice feedback, not an official ranking or a 100-point score. The pilot scores original-task completion; fixed-run numerical errors are separate feedback, not tuning points; see [Assessment and practice feedback](../../ETC/course_materials/ASSESSMENT.md).


In [ ]:
if not RUN_COMPLETED.get(2, False):
    raise RuntimeError("The current Level 2 training run has not completed.")
result_dir = RUN_DIRS[2]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Check your understanding

- Which terms transport temperature, smooth it, add heat, or exchange heat between fields?
- Why does an exchange term vanish from the sum of the two equations?
- Why is a zero-temperature prediction not a valid solution, even if its PDE residual and late-time absolute error are small?
- Change one physical coefficient, predict its effect, then save and run the experiment. Which independent reference remains valid?

In Challenges 1–3, each PINN run learns one configured problem. [Challenge 4 · Neural Operators](../04_neural_operators/Challenge_4_Neural_Operators.ipynb) learns a map from an entire forcing field to its solution field. [Start Here](../../Start_Here.ipynb).

Adapted from the OpenHackathons materials. [License](../../LICENSE).


## Submit your code

Save the edited `.py` files first. Run the cell below. Before your first submission, enter your **Nickname** and click **Register nickname**. This name is shared by Challenges 1-4 and appears on the public scoreboard. Then select the completed Levels and click **Submit code**. Queue status and scores appear here. Running the cell or Run All does not submit code.

The notebook submits `student_equations`, `student_parameters`, `student_conditions`, and `student_solution`. Your conditions, coefficients, and geometry are part of the answer, not replaced with instructor defaults. The bounded language accepts arithmetic, local assignments, literal dictionaries/tuples, `.diff()`, `sin()`, `cos()` and `exp()`; it does not execute arbitrary Python.

Each Level has 100 pilot completion points, split across its required components. Fixed-budget numerical errors are feedback, not extra points for optimizer tuning. Fully correct submissions share the same score and rank. Old PDE-only submissions must be updated to this contract.

Include all Levels you want counted in this attempt. Omitted Levels earn zero; separate attempts are not combined. Your highest complete attempt remains on the board. Pilot points are provisional, not official event scores.


In [ ]:
# Run All only opens these controls. Submission requires a button click.
show_submission_controls("3", LAB_DIR,
    levels=(1,), reference=USE_REFERENCE, judge_url=JUDGE_URL)


--- 

Further resources: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
